# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# Table Silver produite au Jour 2 (matchs nettoyés, typés et enrichis)
SILVER_MAIN_PATH = "/lakehouse/silver/matches"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-118b88d7-3442-4887-b2ff-013ad6846e93;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 230ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

Lignes disponibles dans v_silver : 238855


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [3]:
# Variable continue principale : le nombre total de buts marqués dans un match
# (fthome + ftaway). C'est la métrique qui alimentera les tables Gold et le dashboard.
df_dist_summary = spark.sql("""
    WITH buts AS (
        SELECT fthome + ftaway AS total_goals
        FROM v_silver
    )
    SELECT
        COUNT(*)                                   AS nb_matchs,
        ROUND(AVG(total_goals), 3)                 AS moyenne,
        ROUND(STDDEV(total_goals), 3)              AS ecart_type,
        ROUND(SKEWNESS(total_goals), 3)            AS skewness,
        PERCENTILE_APPROX(total_goals, 0.25)       AS p25,
        PERCENTILE_APPROX(total_goals, 0.50)       AS p50_mediane,
        PERCENTILE_APPROX(total_goals, 0.75)       AS p75,
        PERCENTILE_APPROX(total_goals, 0.75)
            - PERCENTILE_APPROX(total_goals, 0.25) AS iqr,
        PERCENTILE_APPROX(total_goals, 0.90)       AS p90,
        PERCENTILE_APPROX(total_goals, 0.95)       AS p95,
        MAX(total_goals)                           AS maximum
    FROM buts
""")

assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)

# Lecture de la distribution complète : combien de matchs pour chaque score total ?
# Cela permet de visualiser la queue de distribution à droite révélée par le skewness.
spark.sql("""
    WITH buts AS (
        SELECT fthome + ftaway AS total_goals
        FROM v_silver
    )
    SELECT
        total_goals,
        COUNT(*)                                                  AS nb_matchs,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)        AS pct,
        ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY total_goals)
              / SUM(COUNT(*)) OVER (), 2)                         AS pct_cumule
    FROM buts
    GROUP BY total_goals
    ORDER BY total_goals
""").show(20, truncate=False)

+---------+-------+----------+--------+---+-----------+---+---+---+---+-------+
|nb_matchs|moyenne|ecart_type|skewness|p25|p50_mediane|p75|iqr|p90|p95|maximum|
+---------+-------+----------+--------+---+-----------+---+---+---+---+-------+
|238855   |2.642  |1.651     |0.603   |1.0|2.0        |4.0|3.0|5.0|6.0|13.0   |
+---------+-------+----------+--------+---+-----------+---+---+---+---+-------+



26/11/18 16:06:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:06:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:06:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:06:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:06:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:06:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 1

+-----------+---------+-----+----------+
|total_goals|nb_matchs|pct  |pct_cumule|
+-----------+---------+-----+----------+
|0.0        |18901    |7.91 |7.91      |
|1.0        |43617    |18.26|26.17     |
|2.0        |58704    |24.58|50.75     |
|3.0        |51785    |21.68|72.43     |
|4.0        |34324    |14.37|86.80     |
|5.0        |18468    |7.73 |94.53     |
|6.0        |8357     |3.50 |98.03     |
|7.0        |3198     |1.34 |99.37     |
|8.0        |1088     |0.46 |99.83     |
|9.0        |310      |0.13 |99.96     |
|10.0       |79       |0.03 |99.99     |
|11.0       |18       |0.01 |100.00    |
|12.0       |5        |0.00 |100.00    |
|13.0       |1        |0.00 |100.00    |
+-----------+---------+-----+----------+



**Justification statistique — distribution du nombre total de buts par match (`fthome + ftaway`)**

| Indicateur | Valeur |
|---|---|
| Nb de matchs | ~238 854 |
| Moyenne | 2,64 buts |
| Écart-type | 1,65 but |
| **Skewness** | **+0,60** |
| P25 / **médiane** / P75 | 1 / **2** / 4 |
| IQR (P75 − P25) | 3 buts |
| P90 / P95 / Max | 5 / 6 / 13 |

**Lecture de l'asymétrie.** Le coefficient d'asymétrie est positif (**+0,60**), donc la distribution est **étalée vers la droite** : elle est bornée à gauche (un match ne peut pas descendre sous 0 but) mais sa queue droite s'étire jusqu'à **13 buts**. C'est la signature d'une variable de comptage (loi de Poisson) et non d'une loi normale : le test de symétrie échoue, les intervalles « moyenne ± 2σ » n'ont donc pas de sens ici (ils donneraient −0,66 but comme borne basse).

**Écart moyenne / médiane.** La moyenne (2,64) est **supérieure à la médiane (2,0)** — relation typique `médiane < moyenne` d'une asymétrie droite. Les ~5,5 % de matchs à 6 buts ou plus tirent la moyenne vers le haut alors qu'ils ne concernent qu'une petite minorité de rencontres. La moyenne décrit donc mal le match « ordinaire » : **plus de la moitié des matchs comptent 2 buts ou moins**, alors que la moyenne suggère presque 3.

**Impact des valeurs extrêmes.** Les scores fleuves (10 à 13 buts) sont de vraies observations, pas des erreurs de saisie : il ne faut pas les supprimer, mais les **neutraliser dans les indicateurs**. Une moyenne par division ou par saison reste sensible à une poignée de matchs à 9+ buts, alors que la médiane et l'IQR sont insensibles à ces points tant qu'ils restent sous 25 % de l'effectif.

**Indicateurs retenus pour le dashboard (Jours 3 & 4).**
- **Médiane (`PERCENTILE_APPROX(col, 0.5)`)** comme indicateur de tendance centrale : robuste et interprétable (« le match typique »).
- **IQR (P75 − P25 = 3)** comme mesure de dispersion à la place de l'écart-type, lui aussi gonflé par la queue droite.
- **P90 / P95** pour qualifier explicitement les matchs à fort volume de buts (seuil de « match spectaculaire » ≥ 5 buts) au lieu de les noyer dans une moyenne.
- La **moyenne reste affichée à côté de la médiane** : leur écart est lui-même une information métier (plus l'écart est grand, plus la division est irrégulière).

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [4]:
GOLD_PARETO_PATH = "/lakehouse/gold/club_goals_pareto"

df_gold_1 = spark.sql("""
    WITH team_matches AS (

        -- Performances des équipes à domicile
        SELECT
            hometeam AS team,
            fthome AS goals_scored,
            homeelo AS elo
        FROM v_silver
        WHERE hometeam IS NOT NULL
          AND fthome IS NOT NULL

        UNION ALL

        -- Performances des équipes à l'extérieur
        SELECT
            awayteam AS team,
            ftaway AS goals_scored,
            awayelo AS elo
        FROM v_silver
        WHERE awayteam IS NOT NULL
          AND ftaway IS NOT NULL
    ),

    club_stats AS (
        SELECT
            team,

            COUNT(*) AS nb_matchs,

            SUM(goals_scored) AS total_goals,

            ROUND(
                AVG(goals_scored),
                3
            ) AS avg_goals_per_match,

            PERCENTILE_APPROX(
                goals_scored,
                0.50
            ) AS median_goals,

            PERCENTILE_APPROX(
                goals_scored,
                0.75
            )
            -
            PERCENTILE_APPROX(
                goals_scored,
                0.25
            ) AS iqr_goals,

            ROUND(
                AVG(elo),
                2
            ) AS avg_elo

        FROM team_matches

        GROUP BY team
    ),

    ranked AS (
        SELECT
            *,

            DENSE_RANK() OVER (
                ORDER BY total_goals DESC
            ) AS goals_rank,

            ROUND(
                100.0
                *
                SUM(total_goals) OVER (
                    ORDER BY total_goals DESC
                    ROWS BETWEEN UNBOUNDED PRECEDING
                    AND CURRENT ROW
                )
                /
                SUM(total_goals) OVER (),
                2
            ) AS cumulative_goals_pct

        FROM club_stats
    )

    SELECT *
    FROM ranked

    ORDER BY goals_rank, team
""")

assert df_gold_1 is not None

(
    df_gold_1
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PARETO_PATH)
)

print(
    "Table Gold écrite :",
    GOLD_PARETO_PATH
)

print(
    "Nombre de clubs :",
    df_gold_1.count()
)

df_gold_1.show(15, truncate=False)

26/11/18 16:07:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 1

Table Gold écrite : /lakehouse/gold/club_goals_pareto


Nombre de clubs : 1235


26/11/18 16:07:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 1

+-------------+---------+-----------+-------------------+------------+---------+-------+----------+--------------------+
|team         |nb_matchs|total_goals|avg_goals_per_match|median_goals|iqr_goals|avg_elo|goals_rank|cumulative_goals_pct|
+-------------+---------+-----------+-------------------+------------+---------+-------+----------+--------------------+
|Barcelona    |942      |2229.0     |2.366              |2.0         |2.0      |1961.94|1         |0.35                |
|Real Madrid  |943      |2160.0     |2.291              |2.0         |2.0      |1959.08|2         |0.7                 |
|Bayern Munich|869      |2129.0     |2.45               |2.0         |2.0      |1926.88|3         |1.03                |
|Celtic       |900      |2126.0     |2.362              |2.0         |2.0      |1624.13|4         |1.37                |
|PSV Eindhoven|850      |2101.0     |2.472              |2.0         |2.0      |1729.25|5         |1.7                 |
|Ajax         |842      |2048.0 

26/11/18 16:07:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:07:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [5]:
GOLD_TIME_PATH = "/lakehouse/gold/goals_evolution_yearly"

df_gold_2 = spark.sql("""
    WITH yearly_stats AS (
        SELECT
            YEAR(matchdate) AS year,

            COUNT(*) AS nb_matches,

            ROUND(
                AVG(fthome + ftaway),
                3
            ) AS avg_goals_per_match,

            SUM(fthome + ftaway) AS total_goals

        FROM v_silver

        WHERE matchdate IS NOT NULL
          AND fthome IS NOT NULL
          AND ftaway IS NOT NULL

        GROUP BY YEAR(matchdate)
    ),

    with_previous_year AS (
        SELECT
            *,

            LAG(avg_goals_per_match, 1)
            OVER (
                ORDER BY year
            ) AS previous_year_avg

        FROM yearly_stats
    )

    SELECT
        year,
        nb_matches,
        total_goals,
        avg_goals_per_match,

        previous_year_avg,

        ROUND(
            avg_goals_per_match - previous_year_avg,
            3
        ) AS avg_goals_difference,

        ROUND(
            100.0
            * (avg_goals_per_match - previous_year_avg)
            / previous_year_avg,
            2
        ) AS evolution_pct

    FROM with_previous_year

    ORDER BY year
""")

assert df_gold_2 is not None

(
    df_gold_2
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TIME_PATH)
)

print("Table Gold écrite :", GOLD_TIME_PATH)

df_gold_2.show(30, truncate=False)

26/11/18 16:13:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Table Gold écrite : /lakehouse/gold/goals_evolution_yearly


26/11/18 16:13:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+----------+-----------+-------------------+-----------------+--------------------+-------------+
|year|nb_matches|total_goals|avg_goals_per_match|previous_year_avg|avg_goals_difference|evolution_pct|
+----+----------+-----------+-------------------+-----------------+--------------------+-------------+
|2000|2794      |7519.0     |2.691              |NULL             |NULL                |NULL         |
|2001|6329      |16937.0    |2.676              |2.691            |-0.015              |-0.56        |
|2002|6173      |16219.0    |2.627              |2.676            |-0.049              |-1.83        |
|2003|3967      |10433.0    |2.63               |2.627            |0.003               |0.11         |
|2004|5534      |14457.0    |2.612              |2.63             |-0.018              |-0.68        |
|2005|6340      |16142.0    |2.546              |2.612            |-0.066              |-2.53        |
|2006|6959      |17399.0    |2.5                |2.546            |-0.046

26/11/18 16:13:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 16:13:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [6]:
GOLD_TOP_CLUB_PATH = "/lakehouse/gold/top_club_by_division_year"

df_gold_3 = spark.sql("""
    WITH team_matches AS (

        -- Point de vue équipe à domicile
        SELECT
            YEAR(matchdate) AS year,
            division,
            hometeam AS team,
            fthome AS goals_for,
            ftaway AS goals_against,

            CASE
                WHEN ftresult = 'H' THEN 3
                WHEN ftresult = 'D' THEN 1
                ELSE 0
            END AS points,

            CASE WHEN ftresult = 'H' THEN 1 ELSE 0 END AS win,
            CASE WHEN ftresult = 'D' THEN 1 ELSE 0 END AS draw,
            CASE WHEN ftresult = 'A' THEN 1 ELSE 0 END AS loss

        FROM v_silver

        WHERE matchdate IS NOT NULL
          AND division IS NOT NULL
          AND hometeam IS NOT NULL
          AND fthome IS NOT NULL
          AND ftaway IS NOT NULL
          AND ftresult IN ('H', 'D', 'A')

        UNION ALL

        -- Point de vue équipe à l'extérieur
        SELECT
            YEAR(matchdate) AS year,
            division,
            awayteam AS team,
            ftaway AS goals_for,
            fthome AS goals_against,

            CASE
                WHEN ftresult = 'A' THEN 3
                WHEN ftresult = 'D' THEN 1
                ELSE 0
            END AS points,

            CASE WHEN ftresult = 'A' THEN 1 ELSE 0 END AS win,
            CASE WHEN ftresult = 'D' THEN 1 ELSE 0 END AS draw,
            CASE WHEN ftresult = 'H' THEN 1 ELSE 0 END AS loss

        FROM v_silver

        WHERE matchdate IS NOT NULL
          AND division IS NOT NULL
          AND awayteam IS NOT NULL
          AND fthome IS NOT NULL
          AND ftaway IS NOT NULL
          AND ftresult IN ('H', 'D', 'A')
    ),

    club_stats AS (
        SELECT
            year,
            division,
            team,

            COUNT(*) AS matches_played,

            SUM(win) AS wins,
            SUM(draw) AS draws,
            SUM(loss) AS losses,

            SUM(goals_for) AS goals_for,
            SUM(goals_against) AS goals_against,

            SUM(goals_for) - SUM(goals_against) AS goal_difference,

            SUM(points) AS points

        FROM team_matches

        GROUP BY
            year,
            division,
            team

        -- Évite les clubs présents sur seulement quelques matchs
        HAVING COUNT(*) >= 10
    ),

    ranked AS (
        SELECT
            *,

            ROW_NUMBER() OVER (
                PARTITION BY year, division
                ORDER BY
                    points DESC,
                    goal_difference DESC,
                    goals_for DESC
            ) AS ranking

        FROM club_stats
    )

    SELECT
        year,
        division,
        team,
        matches_played,
        wins,
        draws,
        losses,
        goals_for,
        goals_against,
        goal_difference,
        points

    FROM ranked

    WHERE ranking = 1

    ORDER BY year, division
""")

assert df_gold_3 is not None

(
    df_gold_3
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TOP_CLUB_PATH)
)

print("Table Gold écrite :", GOLD_TOP_CLUB_PATH)
print("Nombre de classements :", df_gold_3.count())

df_gold_3.show(30, truncate=False)

Table Gold écrite : /lakehouse/gold/top_club_by_division_year


Nombre de classements : 775


+----+--------+-------------+--------------+----+-----+------+---------+-------------+---------------+------+
|year|division|team         |matches_played|wins|draws|losses|goals_for|goals_against|goal_difference|points|
+----+--------+-------------+--------------+----+-----+------+---------+-------------+---------------+------+
|2000|B1      |Club Brugge  |17            |16  |0    |1     |55.0     |10.0         |45.0           |48    |
|2000|D1      |Schalke 04   |18            |10  |4    |4     |37.0     |19.0         |18.0           |34    |
|2000|D2      |Nurnberg     |18            |13  |3    |2     |39.0     |16.0         |23.0           |42    |
|2000|E0      |Man United   |21            |14  |5    |2     |48.0     |15.0         |33.0           |47    |
|2000|E1      |Fulham       |24            |19  |4    |1     |60.0     |17.0         |43.0           |61    |
|2000|E2      |Millwall     |24            |15  |3    |6     |50.0     |24.0         |26.0           |48    |
|2000|F1  

In [7]:
print("Compaction de la table Silver...")

optimize_result = spark.sql(
    f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`"
)

optimize_result.show(truncate=False)

print("OPTIMIZE terminé sur :", SILVER_MAIN_PATH)

Compaction de la table Silver...


+---------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                         |metrics                                                                                                                                                                                       |
+---------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/matches|{1, 19, {10902112, 10902112, 1.0902112E7, 1, 10902112}, {367724, 1020964, 694782.2105263158, 19, 13200862}, 1, NULL, NULL, 1, 19, 0, false, 0, 0, 1795076890658, 0, 12, 0, NULL, NULL, 55, 32}|
+-------------------------------------------

### Interprétation

La commande `OPTIMIZE` permet de compacter plusieurs petits fichiers Parquet en fichiers plus volumineux.

Un grand nombre de petits fichiers peut ralentir les traitements Spark, car Spark doit ouvrir et lire beaucoup de fichiers séparément.

Cela augmente également la quantité de métadonnées gérées par le NameNode HDFS, car chaque fichier et chaque bloc doit être référencé en mémoire.

La compaction permet donc de réduire le nombre de fichiers, de limiter la charge sur le NameNode et d'améliorer les performances de lecture.